# Water use analysis

In [ ]:
# basic
from pathlib import Path
import pandas as pd
import numpy as np

# spatial
import geopandas as gpd
import xarray as xr
from src.utils import load_config, extract_timeseries

cfg = load_config()
path_data_raw = Path(cfg["dir"]["data"])

## Data

In [ ]:
common_period = slice("2000-01-01", "2010-12-31") # common period for all datasets

# metadata (basins)
metadata_hydroatlas = gpd.read_file("data/hydro/basins_hydroATLAS_lev08.gpkg")

In [ ]:
# water use data from Huang et al. (only consumptive use in mm/month)
ds_cons_total = xr.open_dataset(path_data_raw / "water_use/Huang_HESS/Water_use_Huang_corrected.nc").cons_total
ds_cons_total = ds_cons_total.sel(date=common_period)
ds_cons_total = extract_timeseries(ds_cons_total, metadata_hydroatlas)
ds_cons_total = ds_cons_total.groupby(ds_cons_total.index.month).mean() # monthly mean cycle

In [ ]:
# water availability (near-natural water availability + what if scenarios)
runoff_hydroatlas_hist = xr.open_dataset("data/hydro/runoff_sim_hydroatlas_hist.nc").q_mm_sim
runoff_hydroatlas_hist = runoff_hydroatlas_hist.expand_dims(whatif_scenario=['historical'])
runoff_hydroatlas_whatif = xr.open_dataset("data/hydro/runoff_sim_hydroatlas_whatif.nc").q_mm_sim
runoff_hydroatlas_whatif = runoff_hydroatlas_whatif.groupby('whatif_scenario').mean() # multi model mean

# merge both datasets
runoff_hydroatlas_all = xr.concat([runoff_hydroatlas_hist, runoff_hydroatlas_whatif], dim='whatif_scenario')
#runoff_hydroatlas_all = runoff_hydroatlas_all.sel(date=common_period)
runoff_hydroatlas_all = runoff_hydroatlas_all.resample(date = "MS").sum()
runoff_hydroatlas_all = runoff_hydroatlas_all.groupby(runoff_hydroatlas_all.date.dt.month).mean()
runoff_hydroatlas_all = runoff_hydroatlas_all.to_series().unstack(["whatif_scenario", "gauge_id"])

## WSI

In [ ]:
# water stress index (WSI) = consumption / natural availability
wsi_scenarios = {
    scenario: ds_cons_total / runoff_hydroatlas_all[scenario]
    for scenario in runoff_hydroatlas_all.columns.get_level_values("whatif_scenario").unique()
}
wsi_scenarios = pd.concat(wsi_scenarios).melt(ignore_index=False, value_name="WSI", var_name="gauge_id")

# deficit = max(0, consumption - natural availability)
deficit_scenarios = {
    scenario: np.maximum(ds_cons_total - runoff_hydroatlas_all[scenario], 0)
    for scenario in runoff_hydroatlas_all.columns.get_level_values("whatif_scenario").unique()
}
deficit_scenarios = pd.concat(deficit_scenarios).melt(ignore_index=False, value_name="deficit", var_name="gauge_id")

# join using gauge_id as part of the index (avoids duplicated gauge_id column)
wsi_idx = wsi_scenarios.set_index("gauge_id", append=True)
deficit_idx = deficit_scenarios.set_index("gauge_id", append=True)[["deficit"]]

df = (
    wsi_idx
    .join(
        deficit_idx,
        how="inner",
    )
    .reset_index()
    .rename(
        columns={
            "level_0": "scenario",
            "level_1": "month",
            "date": "month",
        }
    )
)

df.to_parquet("data/hydro/water_stress.parquet")